# Selecting MiniSEED Files from QuakeMigrate

Welcome to the next step of our pipeline!

Right now, we have a massive folder called `unpack_top_300_miniseed_raw/` filled with thousands of extracted waveform files. We also have a very neat spreadsheet (`filtered_picks_organized.csv`) that tells us exactly *when* an earthquake wave arrived at a specific station.

**Our Goal:** We need to find the specific MiniSEED files that actually recorded those earthquake wave arrivals, and copy them into a new folder (`selected_quick_migrate_mseed/`). We will feed this curated folder later.

### Step 1: Setting up our workspace
We import our tools. We use `csv` to read our spreadsheet, and `shutil` which is Python's tool for copying files safely without corrupting them.

In [9]:
import os
import glob
import csv
import shutil
from collections import defaultdict
from obspy import UTCDateTime

mseed_dir = 'unpack_top_300_miniseed_raw'
csv_file = 'filtered_picks_organized.csv'
output_dir = 'selected_quick_migrate_mseed'

# Create the output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created output directory: {output_dir}")
else:
    print(f"Output directory '{output_dir}' is ready.")

Output directory 'selected_quick_migrate_mseed' is ready.


### Step 2: Building a Fast Index
Imagine trying to find a specific topic in a 6,000-page textbook by flipping through every page one by one. It would take forever! It's much faster to use the **Index** at the back of the book.

Because we have thousands of files and hundreds of rows in our CSV, asking Python to scan the entire hard drive for every single row would be painfully slow.

Instead, we scan the hard drive *once* at the very beginning. We read the neat filenames we created earlier (like `2E.TJTJ.HHZ.20200724T212502.mseed`), extract the station and timestamps, and organize them into a quick-lookup dictionary (`mseed_index`).

In [10]:
print(f"Indexing MiniSEED files in {mseed_dir}/ ...")
mseed_files = glob.glob(os.path.join(mseed_dir, '*.mseed'))

# defaultdict creates an empty list automatically if a station isn't in the dictionary yet
mseed_index = defaultdict(list)

for filepath in mseed_files:
    basename = os.path.basename(filepath)
    parts = basename.split('.')
    
    # Ensure the filename matches our expected format
    if len(parts) >= 6:
        station = parts[1]
        start_str = parts[3]
        end_str = parts[4]
        
        try:
            # Convert string times to UTCDateTime so we can do math with them later
            start_time = UTCDateTime(start_str)
            end_time = UTCDateTime(end_str)
            
            # Add this file to our quick-lookup index under its station name
            mseed_index[station].append({
                'filepath': filepath,
                'basename': basename,
                'start_time': start_time,
                'end_time': end_time
            })
        except Exception as e:
            pass # Skip files with bad names

print(f"Indexed {len(mseed_files)} files across {len(mseed_index)} unique stations.")

Indexing MiniSEED files in unpack_top_300_miniseed_raw/ ...
Indexed 6300 files across 14 unique stations.


### Step 3: Scanning the CSV & Copying Matches
Now we open our CSV file and read it row by row.

For each row, we:
1. Grab the `station` and `pick_time`.
2. Instantly look up all files for that `station` using our index.
3. Check the math: Is `start_time <= pick_time <= end_time`?
4. If yes, we copy the file! We also use a `set()` to remember which files we copied so we don't accidentally copy the same file twice.

In [11]:
print(f"\nScanning {csv_file} for wave arrival matches...")
copied_files = set() # A set is like a list, but it automatically prevents duplicates!
match_count = 0
row_count = 0

with open(csv_file, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        row_count += 1
        station = row['station']
        
        try:
            pick_time = UTCDateTime(row['pick_time'])
        except Exception:
            continue
            
        # 1. Quick Lookup
        if station in mseed_index:
            for file_info in mseed_index[station]:
                
                # 2. Check the math
                if file_info['start_time'] <= pick_time <= file_info['end_time']:
                    
                    # 3. Prevent duplicates
                    if file_info['basename'] not in copied_files:
                        src = file_info['filepath']
                        dst = os.path.join(output_dir, file_info['basename'])
                        
                        # 4. Copy! (We use copy2 to preserve timestamps)
                        shutil.copy2(src, dst)
                        copied_files.add(file_info['basename'])
                        match_count += 1

print("--- SUMMARY ---")
print(f"Total CSV rows checked: {row_count}")
print(f"Total unique .mseed files successfully copied: {match_count}")


Scanning filtered_picks_organized.csv for wave arrival matches...
--- SUMMARY ---
Total CSV rows checked: 399
Total unique .mseed files successfully copied: 1110


### Step 4: Verification!
Let's double check our work. Did the math actually work?
We will pick the very first row from the CSV, look at the exact timestamps, and see if the files our script selected actually make logical sense.

In [12]:
print("--- VISUAL VERIFICATION (1 EXAMPLE) ---")

with open(csv_file, 'r') as f:
    reader = csv.DictReader(f)
    # We just grab the very first row using next()
    first_row = next(reader)
    
station = first_row['station']
pick_time = UTCDateTime(first_row['pick_time'])

print(f"CSV Info Used:")
print(f"  - Station:   {station}")
print(f"  - Pick Time: {pick_time}")

if station in mseed_index:
    print(f"\nOut of {len(mseed_index[station])} total files for {station}, here is what we selected:")
    
    for file_info in mseed_index[station]:
        if file_info['start_time'] <= pick_time <= file_info['end_time']:
            print(f"  ✅ {file_info['basename']}")
            print(f"      Because {file_info['start_time']} <= {pick_time} <= {file_info['end_time']}")
else:
    print(f"No files available for station {station}")

--- VISUAL VERIFICATION (1 EXAMPLE) ---
CSV Info Used:
  - Station:   DRSC
  - Pick Time: 2020-03-09T02:49:18.086651Z

Out of 462 total files for DRSC, here is what we selected:
  ✅ 2E.DRSC.HH1.20200309T024911.20200309T024953.mseed
      Because 2020-03-09T02:49:11.000000Z <= 2020-03-09T02:49:18.086651Z <= 2020-03-09T02:49:53.000000Z
  ✅ 2E.DRSC.HHZ.20200309T024911.20200309T024952.mseed
      Because 2020-03-09T02:49:11.000000Z <= 2020-03-09T02:49:18.086651Z <= 2020-03-09T02:49:52.000000Z
  ✅ 2E.DRSC.HH2.20200309T024911.20200309T024953.mseed
      Because 2020-03-09T02:49:11.000000Z <= 2020-03-09T02:49:18.086651Z <= 2020-03-09T02:49:53.000000Z


---
**Great Job!**
You've learned how to cross-reference data sources programmatically. Instead of doing O(N*M) checks (which is very slow), you built a fast lookup index. Your data is now perfectly curated and ready!